# **OpenPyTEA** Walkthrough — Part 1: Defining Equipment

This notebook is **Part 1** of the OpenPyTEA walkthrough series:

1. **Part 1: Defining Equipment** (this notebook)
2. [Part 2: Creating the Plant](part_2_plant.ipynb)
3. [Part 3: Cost Analysis and Sensitivity](part_3_analysis.ipynb)
4. [Part 4: Monte Carlo Uncertainty Analysis](part_4_monte_carlo.ipynb)
5. [Part 5: TEA Using Configuration Files](part_5_configuration_files.ipynb)

## 🧭 Overview

This walkthrough series demonstrates the full workflow of **OpenPyTEA**, an open-source Python toolkit for transparent and reproducible **techno-economic assessment (TEA)** of chemical and energy systems.

Across its five parts, you will learn how to:
1. Define process equipment and estimate their purchased and direct (installed) costs.
2. Assemble a plant configuration and calculate capital and operating expenditures.
3. Evaluate economic performance metrics such as **NPV**, **IRR**, **payback time**, and **levelized cost (LCOP)**.
4. Visualize **cost breakdowns** of equipment costs, fixed capital, as well as variable and fixed OPEX.
5. Visualize the project's **cumulative cash flow diagram**, including its maximum investment and pay-back point.
6. Perform **sensitivity** and **uncertainty analyses** and plotting the results.

The series is intended for both researchers and students, providing a practical, hands-on introduction to techno-economic modeling directly in Python.

The parts build on each other, but every notebook can also be run on its own: parts 2 to 4 start with a short **Setup** cell that recreates the objects from the preceding parts.

## ⚙️ Setup & Imports

Before getting started, make sure that **OpenPyTEA** is installed. You can install it with:

In [1]:
# The [ipython] extra adds ipywidgets for the Monte Carlo progress bar in notebooks.
# Development version of OpenPyTEA, not yet on PyPI:
# %pip install "openpytea[ipython] @ git+https://github.com/pbtamarona/OpenPyTEA"
# Stable version of OpenPyTEA, available on PyPI:
# %pip install "openpytea[ipython]"

The following modules will be used:

| Module      | Description                                                                                                                                                |
| ----------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `equipment` | Defines individual process units, cost correlations, and inflation adjustment methods for estimating purchased equipment costs.                            |
| `plant`     | Combines equipment objects with plant-wide economic assumptions to calculate CAPEX, OPEX, cash flow, and TEA metrics such as NPV, LCOP, ROI, PBT, and IRR. |
| `analysis`  | Prepares data for cost breakdowns, sensitivity analysis, tornado diagrams, and Monte Carlo uncertainty analysis.                                           |
| `plotting`  | Generates visualizations for cost breakdowns, sensitivity results, tornado plots, and Monte Carlo distributions.                                           |
| `io`        | Loads equipment, plant, and analysis configurations from JSON files and supports running TEA workflows from structured input files.                        |

## 🧱 Define Equipment

Each process unit (e.g., compressor, heat exchanger, reactor) is represented by an `Equipment` object.  
You can specify the **category**, **type**, **material**, and optionally provide custom cost data or scaling factors.

To define Equipment objects, you will need the equipment module:

In [2]:
from openpytea import Equipment

The equipment module automatically:
1. Selects the appropriate cost correlation from `data/cost_correlations.csv` based on the equipment `category` and `type`.
2. Estimates the purchased and direct costs. Note that all cost correlations in the database are in (or adjusted to) US dollars.
3. Adjusts costs for inflation using CEPCI data to the 2024 cost year.
4. Stores all technical and economic attributes for plant-level aggregation.
5. If the equipment type is not found in the database, the user must specify its `purchased_cost`.
6. Groups sub-components into a single `CompositeEquipment` line item (usage example #10).

**Purchased cost** is estimated via cost correlation or specified directly. Six correlation forms are supported:
1. **Offset power-law**: $C_p = a + b \cdot S^n$
2. **Log-log quadratic**: $\log_{10}(C_p) = K_1 + K_2 \cdot \log_{10}(S) + K_3 \cdot [\log_{10}(S)]^2 + K_4 \cdot [\log_{10}(S)]^3 + K_5 \cdot [\log_{10}(S)]^4$
3. **Ln-ln quadratic**: $\ln(C_p) = K_1 + K_2 \cdot \ln(S) + K_3 \cdot [\ln(S)]^2 + K_4 \cdot [\ln(S)]^3 + K_5 \cdot [\ln(S)]^4$ (same as log-log quadratic but base-$e$; $K_4$/$K_5$ are optional and default to 0)
4. **Power-sizing**: $C_p = C_0 \cdot (S / S_0)^f$
5. **Exponential**: $C_p = a \cdot \exp(b \cdot S)$
6. **2-var power-law**: $C_p = a + b \cdot S_1^{n} \cdot S_2^{n_2}$ ($a$ is typically 0) — for equipment priced off two independent size parameters; pass `param=(S1, S2)` as a tuple

where $C_p$ is the purchased equipment cost, $S$ (or $S_1$, $S_2$) is the size/capacity parameter, and $a$, $b$, $n$, $n_2$, $K_1$–$K_5$, $S_0$, $C_0$, $f$ are correlation coefficients.

**Direct cost** covers installed costs beyond the purchased equipment (erection, piping, electrical, instrumentation, civil, structural, and lagging):
$$C_D = C_p \left[ (1 + f_p) \cdot f_m + \left( f_{er} + f_{el} + f_i + f_c + f_s + f_l \right) \right]$$
*Source: Towler, G.; Sinnott, R. Chemical Engineering Design; Elsevier, 2022. https://doi.org/10.1016/C2019-0-02025-0*

where $C_D$ is the direct cost, $f_m$ is the material factor, and $f_p$, $f_{er}$, $f_{el}$, $f_i$, $f_c$, $f_s$, $f_l$ are the piping, erection, electrical, instrumentation, civil, structural, and lagging factors. Usage example #8 presents the default material and construction factor values used in OpenPyTEA.

Here are the mandatory and optional input for `Equipment` objects: 

| **Input Parameter**                   | **Description**                                                                                                                                                                                              | **Example**                                               |
|--------------------------------------|---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|-----------------------------------------------------------|
| `name`                               | User-defined name or label for the equipment instance.                                                                                                                                                         | `"Reactor R-101"`, `"Pump P-201"`                         |
| `param`                              | Primary sizing or design variable used in cost correlations (e.g., volume, area, flow rate). Units depend on the equipment type (see `data/cost_correlations.csv`). Pass a 2-element tuple/list `(S1, S2)` for two-parameter forms (e.g., `"2-var power-law"`). | `50` (m³), `800` (m²), `5000` (kW), `(24, 150)`           |
| `process_type`                       | Type of process handled by the equipment, which determines installation and auxiliary cost factors. Must be one of `"Solids"`, `"Fluids"`, `"Mixed"`, or `"Electrical"`.                                        | `"Fluids"`                                                |
| `category`                           | Equipment category. Must match the `category` field (if available) in `data/cost_correlations.csv` to use the corresponding cost correlation.                                                                  | `"Pump"`, `"Heat Exchanger"`, `"Compressor"`, `"Reactor"` |
| `type` *(optional)*                  | Equipment subtype. Must match the `type` field (if available) in `data/cost_correlations.csv` to use the corresponding cost correlation.                                                                        | `"Centrifugal"`, `"Shell-and-tube"`                       |
| `material` *(optional)*              | Material of construction. Affects installed cost via material multipliers. Defaults to `"Carbon steel"`.                                                                                                       | `"316 stainless steel"`, `"Aluminum"`, `"Monel"`          |
| `num_units` *(optional)*             | Number of identical units. Automatically increased if the required capacity exceeds the correlation's valid range.                                                                                             | `2`                                                       |
| `purchased_cost` *(optional)*        | Manually override the calculated purchased cost. If provided, the correlation and `param` are ignored.                                                                                                          | `1.5e6`                                                   |
| `cost_func` *(optional)*             | Explicitly specify a correlation key from `data/cost_correlations.csv`, overriding automatic category/type matching.                                                                                            | `"HX_shell_tube_01"`                                      |
| `target_year` *(optional)*           | Target year for inflation adjustment via CEPCI. Defaults to `2024`.                                                                                                                                            | `2025`                                                    |
| `erection_factor` *(optional)*       | Override the equipment erection factor (foundations, minor structural work). Defaults to the value for the selected `process_type`.                                                                             | `0.4`                                                     |
| `piping_factor` *(optional)*         | Override the piping factor (insulation and painting). Defaults to the value for the selected `process_type`.                                                                                                    | `0.5`                                                     |
| `instrumentation_factor` *(optional)*| Override the instrumentation & controls factor. Defaults to the value for the selected `process_type`.                                                                                                          | `0.25`                                                    |
| `electrical_factor` *(optional)*     | Override the electrical factor (power and lighting). Defaults to the value for the selected `process_type`.                                                                                                     | `0.15`                                                    |
| `civil_factor` *(optional)*          | Override the civil factor (buildings and structures). Defaults to the value for the selected `process_type`.                                                                                                    | `0.2`                                                     |
| `structural_factor` *(optional)*     | Override the structural steel factor. Defaults to the value for the selected `process_type`.                                                                                                                    | `0.1`                                                     |
| `lagging_factor` *(optional)*        | Override the lagging & painting factor. Defaults to the value for the selected `process_type`.                                                                                                                  | `0.05`                                                    |
| `material_factor` *(optional)*       | Override the material cost multiplier. Defaults to the multiplier for the selected `material`.                                                                                                                  | `1.4`                                                     |

Multiple equipment objects can be defined and later combined into a complete process plant using the `Plant` class (next section).

### Equipment Class – Usage Examples

Below are practical examples demonstrating all key functionalities of the `Equipment` class.

**1. Automatic Correlation Lookup**

Uses `category` and `type` to fetch cost correlations automatically and apply CEPCI inflation. If there is there is multiple cost correlation with the same `category` and `type`, **OpenPyTEA** selects the first matching correlation listed in the database.

In [3]:
hx = Equipment(
    name="HX-101",
    param=900,                      # e.g., m² area
    process_type="Fluids",
    category="Heat Exchangers",
    type="U-tube shell & tube",
    material="316 stainless steel"
)

print(hx)

# To fetch the purchase and direct installation costs you can use:
print(hx.purchased_cost)
print(hx.direct_cost)

Name=HX-101, Category=Heat Exchangers, Sub-type=U-tube shell & tube, Material=316 stainless steel, Process Type=Fluids, Parameter=900, Number of units=1, Purchased Cost=315825.28434063436, Direct Cost=1181186.5634339727)
315825.28434063436
1181186.5634339727


**2. Using an Explicit `cost_func` Key**

Bypass automatic lookup and use a known cost-correlation key directly.

In [4]:
comp1 = Equipment(
    name='Comp-1', 
    process_type='Fluids', 
    material='Carbon steel', 
    param=1, # MW
    category='Compressors, fans, & Blowers', 
    type='Compressor, centrifugal',
    cost_func = 'co2_compressor_manzolini_2011'
)

print(comp1)

Name=Comp-1, Category=Compressors, fans, & Blowers, Sub-type=Compressor, centrifugal, Material=Carbon steel, Process Type=Fluids, Parameter=1, Number of units=1, Purchased Cost=3387744.1024415228, Direct Cost=10840781.127812874)


**3. Manually Defined `purchased_cost`**

Skip correlation and inflation calculations by supplying your own purchased cost and cost year (optional).

In [5]:
dryer = Equipment(
    name="Rotary Dryer D-301",
    param=0,                        # ignored
    process_type="Solids",
    category="Dryer",
    material="Carbon steel",
    purchased_cost=1_500_000,        # already in target-year money
    cost_year=2021,
)
print(dryer)

Name=Rotary Dryer D-301, Category=Dryer, Sub-type=None, Material=Carbon steel, Process Type=Solids, Parameter=None, Number of units=1, Purchased Cost=1693002.2573363432, Direct Cost=4232505.643340858)


**4. Auto-Parallelization for Oversized Equipment**

If `param` exceeds the upper bound in the correlation, the system automatically creates multiple parallel units.

In [6]:
comp2 = Equipment(
    name="Air Compressor",
    param=50_000,                   # exceeds upper_parallel → auto-splits
    process_type="Fluids",
    category='Compressors, fans, & blowers', 
    type='Compressor, centrifugal',
)
print(comp2)
print(comp2.num_units)

Name=Air Compressor, Category=Compressors, fans, & blowers, Sub-type=Compressor, centrifugal, Material=Carbon steel, Process Type=Fluids, Parameter=50000, Number of units=2, Purchased Cost=26973146.35573345, Direct Cost=86314068.33834705)
2


 **5. Inflation Adjustment to a Custom Year**

Calculate cost for a different inflation-adjusted year using CEPCI.

In [7]:
hx_2020 = Equipment(
    name="HX E-102",
    param=850,
    process_type="Fluids",
    category="Heat Exchangers",
    type="U-tube shell & tube",
    material="316 stainless steel",
    target_year=2020
)
print(hx_2020)

Name=HX E-102, Category=Heat Exchangers, Sub-type=U-tube shell & tube, Material=316 stainless steel, Process Type=Fluids, Parameter=850, Number of units=1, Purchased Cost=221775.1706670056, Direct Cost=829439.138294601)


Equipment costs are adjusted with CEPCI as follows:

$$C_{\text{adjusted}} = C_{\text{base}} \times \frac{\text{CEPCI}_{\text{target}}}{\text{CEPCI}_{\text{base}}}$$

Where $C_{\text{adjusted}}$ is the equipment cost in the target year (USD), $C_{\text{base}}$ is equipment cost in the base year (USD), $\text{CEPCI}_{\text{target}}$ is the CEPCI value for the target year, $\text{CEPCI}_{\text{base}}$ is the CEPCI for the base year

**6. Forcing a Specific Number of Units**

Set `num_units` to price several identical units: the correlation is evaluated at `param` for one unit and the result is multiplied by `num_units`. (A direct `purchased_cost` is not multiplied; it is taken as the total for all units.)

In [ ]:
fridge = Equipment(
    name="Refrigerator R-201",
    param=180,
    process_type="Fluids",
    category="Utilities",
    type="Packaged mechanical refrigerator",
    num_units=3                    # manually fix number of units
)
print(fridge)

Name=Refrigerator R-201, Category=Utilities, Sub-type=Packaged mechanical refrigerator, Material=Carbon steel, Process Type=Fluids, Parameter=180, Number of units=3, Purchased Cost=1737742.212831217, Direct Cost=5560775.081059895)
Name=Refrigerator R-201, Category=Utilities, Sub-type=Packaged mechanical refrigerator, Material=Carbon steel, Process Type=Fluids, Parameter=180, Number of units=1, Purchased Cost=579247.4042770724, Direct Cost=1853591.6936866317)


**7. Effect of Material and Process Type on Direct Cost**

Demonstrates how material and process factors modify the direct (installed) cost.

In [9]:
mixer_cs = Equipment(
    name="Agitator M-101 (CS)",
    param=100,
    process_type="Fluids",
    category="Agitators, blenders, & mixers",
    type="Propeller mixer",
    material="Carbon steel"
)
print(mixer_cs)

mixer_alloy = Equipment(
    name="Agitator M-102 (Alloy)",
    param=10,
    process_type="Fluids",
    category="Agitators, blenders, & mixers",
    type="Propeller mixer",
    material="Hastelloy C"
)
print(mixer_alloy)

mixer_solids = Equipment(
    name="Agitator M-101 (Mixed)",
    param=100,
    process_type="Solids",
    category="Agitators, blenders, & mixers",
    type="Propeller mixer",
    material="Carbon steel"
)
print(mixer_solids)

Name=Agitator M-101 (CS), Category=Agitators, blenders, & mixers, Sub-type=Propeller mixer, Material=Carbon steel, Process Type=Fluids, Parameter=100, Number of units=2, Purchased Cost=248965.4645525466, Direct Cost=796689.4865681492)
Name=Agitator M-102 (Alloy), Category=Agitators, blenders, & mixers, Sub-type=Propeller mixer, Material=Hastelloy C, Process Type=Fluids, Parameter=10, Number of units=1, Purchased Cost=43106.47572056963, Direct Cost=180616.13326918677)
Name=Agitator M-101 (Mixed), Category=Agitators, blenders, & mixers, Sub-type=Propeller mixer, Material=Carbon steel, Process Type=Solids, Parameter=100, Number of units=2, Purchased Cost=248965.4645525466, Direct Cost=622413.6613813664)


Here are some of the available material options and their corresponding installation cost multipliers:
- **Carbon steel** — 1.00  
- **Aluminum** — 1.07  
- **Bronze** — 1.07  
- **Cast steel** — 1.10  
- **Stainless steel** — 1.30*  
- **304 stainless steel** — 1.30  
- **316 stainless steel** — 1.30  
- **321 stainless steel** — 1.50  
- **Hastelloy C** — 1.55  
- **Monel** — 1.65  
- **Nickel** — 1.70  
- **Inconel** — 1.70  
*not from the original table — added here, equal to SS304

*Source: Towler, G.; Sinnott, R. Chemical Engineering Design; Elsevier, 2022. https://doi.org/10.1016/C2019-0-02025-0*

The installation cost factors vary by process type. Here are the default values used when no override is provided:

| Factor | **Solids** | **Fluids** | **Mixed** | **Electrical** |
|---|:---:|:---:|:---:|:---:|
| Erection (`fer`) | 0.60 | 0.30 | 0.50 | 0.40 |
| Piping (`fp`) | 0.20 | 0.80 | 0.60 | 0.10 |
| Instrumentation (`fi`) | 0.20 | 0.30 | 0.30 | 0.70 |
| Electrical (`fel`) | 0.15 | 0.20 | 0.20 | 0.70 |
| Civil (`fc`) | 0.20 | 0.30 | 0.30 | 0.20 |
| Structural steel (`fs`) | 0.10 | 0.20 | 0.20 | 0.10 |
| Lagging & painting (`fl`) | 0.05 | 0.10 | 0.10 | 0.10 |

*Source: Towler, G.; Sinnott, R. Chemical Engineering Design; Elsevier, 2022. https://doi.org/10.1016/C2019-0-02025-0*

**8. Automatic Default Material Resolution**

Leaving `material` unset resolves it from the matched correlation's own `default material` in `cost_correlations.csv`, with a material factor of 1.0 — the correlation's quoted cost already prices in that material, so no extra multiplier is applied. Passing a *different* material instead rescales the `material_factors` table value relative to the correlation's own default, rather than the usual carbon steel baseline.

In [10]:
# This batch centrifuge correlation defaults to "Stainless steel"
centrifuge = Equipment(
    name="Centrifuge C-101",
    param=30,                        # bowl diameter, in
    process_type="Fluids",
    category="Centrifuges",
    type="Batch, bottom-drive, vertical basket",
)
print(f"material        : {centrifuge.material}")
print(f"material_factor : {centrifuge.material_factor}")

# Explicitly requesting Inconel rescales the factor relative to the
# correlation's own stainless steel basis: 1.70 / 1.30 ≈ 1.31, not 1.70
centrifuge_inconel = Equipment(
    name="Centrifuge C-101 (Inconel)",
    param=30,
    process_type="Fluids",
    category="Centrifuges",
    type="Batch, bottom-drive, vertical basket",
    material="Inconel",
)
print(f"material_factor : {centrifuge_inconel.material_factor:.3f}")

material        : Stainless steel
material_factor : 1.0
material_factor : 1.308


**9. Overriding Installation Cost Factors**

By default, the installation cost factors (, , etc.) and the  are looked up from built-in tables based on  and . You can override any of them individually — for example, when using project-specific data or literature values that differ from the defaults.

The default factor tables are accessible as class attributes:
-  — factors per process type (, , , )
-  — material multipliers

Any factor not overridden falls back to its table default.

In [11]:
# Default factors for the Fluids process type
print(Equipment.process_factors["Fluids"])
print(Equipment.material_factors["316 stainless steel"])
# Override only piping_factor and material_factor; all other factors use Fluids defaults
reactor = Equipment(
    name="Reactor R-101",
    param=50,
    process_type="Fluids",
    category="Reactors",
    type="Glass-lined agitated",
    material="316 stainless steel",
    piping_factor=0.95,         # custom value instead of default 0.8
    material_factor=1.4,        # custom value instead of default 1.3
)
print(reactor)
print(f"piping_factor: {reactor.piping_factor}")
print(f"material_factor: {reactor.material_factor}")

{'fer': 0.3, 'fp': 0.8, 'fi': 0.3, 'fel': 0.2, 'fc': 0.3, 'fs': 0.2, 'fl': 0.1}
1.3
Name=Reactor R-101, Category=Reactors, Sub-type=Glass-lined agitated, Material=316 stainless steel, Process Type=Fluids, Parameter=50, Number of units=2, Purchased Cost=965658.1893445571, Direct Cost=3988168.321993021)
piping_factor: 0.95
material_factor: 1.4


**10. Composite Equipment (Sub-components)**

Some equipment items are assemblies: a pressure-swing adsorption (PSA) unit is a set of adsorber vessels filled with layers of different adsorbents, a compressor train is a compressor plus its driver, a reactor carries a catalyst charge. `CompositeEquipment` assembles one equipment line item from sub-components. Every sub-component is an ordinary `Equipment` (or another `CompositeEquipment`, nesting is allowed), so it brings its own cost correlation or user-defined purchased cost, its own material factor and its own installation factors. Several identical units of a component are priced by setting `num_units` on the component itself, exactly as for stand-alone equipment.

The composite exposes the same attributes as an `Equipment` (`purchased_cost`, `direct_cost`, `num_units`, ...), so it goes into a plant's equipment list like any other item and counts as a single process step in the operator estimate. `category` and `type` are free labels used for reporting.

In [12]:
from openpytea import CompositeEquipment

vessel = Equipment(
    name="Adsorber vessel",
    param=12.0,                        # volume of ONE vessel, m^3
    process_type="Fluids",
    category="Pressure vessels",
    type="Vertical",
    material="304 stainless steel",
    num_units=4,                       # four identical vessels
)
zeolite = Equipment(
    name="Zeolite 5A layer",
    param=4 * 250.0,                   # total bulk volume across the vessels, ft^3
    process_type="Solids",
    category="Packings & adsorbents",
    type="Molecular sieves",
)
carbon = Equipment(
    name="Activated carbon layer",
    param=None,
    process_type="Solids",
    category="Packings & adsorbents",
    type="Activated carbon",
    purchased_cost=38_000.0,           # vendor quote instead of a correlation
    cost_year=2021,
)

psa = CompositeEquipment(
    name="PSA",
    process_type="Fluids",
    components=[vessel, zeolite, carbon],
    category="Adsorption",
    type="Pressure-swing adsorber",
)
print(psa)

Name=PSA, Category=Adsorption, Sub-type=Pressure-swing adsorber, Process Type=Fluids, Installation=component, Number of units=1, Purchased Cost=266464.6703841906, Direct Cost=794761.1432116599)
    - Adsorber vessel (x4): Purchased Cost=103709.2477832123, Direct Cost=387872.586709214
    - Zeolite 5A layer (x1): Purchased Cost=119866.03208179094, Direct Cost=299665.08020447736
    - Activated carbon layer (x1): Purchased Cost=42889.390519187364, Direct Cost=107223.47629796842

`breakdown()` tabulates every leaf component: `purchased_each` is the cost of one unit, while `purchased_total` and `direct_total` cover all `num_units` of that component.

In [13]:
psa.breakdown().round(0)

,component,category,type,material,param,num_units,purchased_each,purchased_total,direct_total
0,Adsorber vessel,Pressure vessels,Vertical,304 stainless steel,12.0,4,25927.0,103709.0,387873.0
1,Zeolite 5A layer,Packings & adsorbents,Molecular sieves,Carbon steel,1000.0,1,119866.0,119866.0,299665.0
2,Activated carbon layer,Packings & adsorbents,Activated carbon,Carbon steel,NaN,1,42889.0,42889.0,107223.0


By default (`installation="component"`) the composite's direct cost is the sum of each part's own direct cost, so the vessels carry the *Fluids* installation factors and the 304SS material factor while the adsorbent layers carry the *Solids* factors. With `installation="composite"` the composite is installed as one item: its own process-type factors are applied once to the total purchased cost.

A vendor quote for the whole skid can also be given via `purchased_cost` and `cost_year`. The quote replaces the component sum (kept as `components_purchased_cost` for reference), is not multiplied by `num_units`, and the composite is then installed as one item.

In [14]:
import pandas as pd

psa_as_one = CompositeEquipment(
    name="PSA", process_type="Fluids",
    components=[vessel, zeolite, carbon],
    installation="composite",
)
psa_quoted = CompositeEquipment(
    name="PSA", process_type="Fluids",
    components=[vessel, zeolite, carbon],
    purchased_cost=650_000.0,          # 2019 quote for the whole skid
    cost_year=2019,
)

pd.DataFrame(
    {
        "installation rule": ["component (default)", "composite", "composite quote"],
        "components purchased": [p.components_purchased_cost for p in (psa, psa_as_one, psa_quoted)],
        "purchased cost": [p.purchased_cost for p in (psa, psa_as_one, psa_quoted)],
        "direct cost": [p.direct_cost for p in (psa, psa_as_one, psa_quoted)],
    }
).set_index("installation rule").round(0)

,components purchased,purchased cost,direct cost
installation rule,,,
component (default),266465.0,266465.0,794761.0
composite,266465.0,266465.0,852687.0
composite quote,266465.0,855967.0,2739095.0


A composite can itself be a component of another composite. The `num_units` of every enclosing composite multiplies along the path, and the breakdown labels show the path.

In [15]:
tail_gas_compressor = Equipment(
    name="Tail-gas compressor",
    param=None,
    process_type="Fluids",
    category="Compressors, fans, & blowers",
    purchased_cost=210_000.0,
    cost_year=2024,
)
train = CompositeEquipment(
    name="Purification train",
    process_type="Fluids",
    components=[psa, tail_gas_compressor],
    num_units=2,                       # two identical trains
)
train.breakdown().round(0)

,component,category,type,material,param,num_units,purchased_each,purchased_total,direct_total
0,PSA / Adsorber vessel,Pressure vessels,Vertical,304 stainless steel,12.0,8,25927.0,207418.0,775745.0
1,PSA / Zeolite 5A layer,Packings & adsorbents,Molecular sieves,Carbon steel,1000.0,2,119866.0,239732.0,599330.0
2,PSA / Activated carbon layer,Packings & adsorbents,Activated carbon,Carbon steel,NaN,2,42889.0,85779.0,214447.0
3,Tail-gas compressor,"Compressors, fans, & blowers",NaN,Carbon steel,NaN,2,210000.0,420000.0,1344000.0


---

**Next:** [Part 2: Creating the Plant](part_2_plant.ipynb) ▶